# Frozen public YOLO + U-Net inference candidate

Derived from Maxim/ektarr's Apache-2.0 Kaggle notebook. The published 0.69 score belongs to the source author; this notebook claims no score until our own submission is accepted and scored. It downloads and verifies the frozen public weights, then performs inference only.

In [ ]:
%pip install -q ultralytics kagglehub

In [ ]:
# ====================================================
# Imports
# ====================================================

import os
import json
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import torch
from ultralytics import YOLO
from pycocotools import mask as mask_utils

warnings.filterwarnings("ignore")

import ultralytics.utils.tqdm as ultra_tqdm
ultra_tqdm.is_noninteractive_console = lambda: True

In [ ]:
# ====================================================
# Config
# ====================================================

class CFG:

    ROOT = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"

    TRAIN_DIR = os.path.join(ROOT, "train", "train_images")
    TEST_DIR = os.path.join(ROOT, "test", "test_images")

    TRAIN_JSON = os.path.join(
        ROOT,
        "train",
        "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
    )

    YOLO_DIR = "/kaggle/working/yolo_dataset"

    DEVICE = 0 if torch.cuda.is_available() else "cpu"
    DEBUG = False

    IMGSZ = 1024
    BATCH_SIZE = 4
    EPOCHS = 1 if DEBUG else 30
    PATIENCE = 5

    MODEL = "yolo11s-seg.pt"

    USE_CATEGORIES = True

    MASK_RATIO = 2

    CONF_THRESHOLD = 0.25
    MIN_COMPONENT_SIZE = 20

    SEED = 42

In [ ]:
import hashlib
import shutil
from pathlib import Path
import kagglehub

PUBLIC_HANDLE = 'ektarr/yolo-unet-solar-filament-segmentation'
EXPECTED = {'best_model.pt': '0721382bc86742d06fdfd9c13f37501f5590cecbda99d5c87d1b77913cdbe365', 'best_refiner.pt': 'b79cf01b614b54117e465ad2b8e76941258435b8ceddbf654aff6bc179624ec2'}
asset_dir = Path(kagglehub.notebook_output_download(PUBLIC_HANDLE))
for name, expected in EXPECTED.items():
    matches = list(asset_dir.rglob(name))
    if len(matches) != 1:
        raise RuntimeError(f"expected exactly one {name}, found {len(matches)}")
    actual = hashlib.sha256(matches[0].read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f"frozen hash mismatch for {name}: {actual}")
    shutil.copy2(matches[0], name)
print("verified frozen public weights", sorted(EXPECTED))


In [ ]:
# ============================================================
# Crop-refiner: light UNet over instances, found by YOLO
# ============================================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights
import numpy as np
import cv2
import random
import os
from tqdm.auto import tqdm

CROP_SIZE = 256
CONTEXT = 1.8
MIN_CROP = 96


def square_bounds(mask, context=CONTEXT, minimum=MIN_CROP):
    height, width = mask.shape
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return 0, 0, width, height
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(np.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return int(x0), int(y0), int(x1), int(y1)


class ConvBlock(nn.Sequential):
    def __init__(self, in_ch, out_ch):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )


class CropUNet(nn.Module):

    def __init__(self, pretrained=True):
        super().__init__()
        enc = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
        self.stem = nn.Sequential(enc.conv1, enc.bn1, enc.relu)
        self.pool, self.layer1, self.layer2, self.layer3, self.layer4 = (
            enc.maxpool, enc.layer1, enc.layer2, enc.layer3, enc.layer4
        )
        self.up4, self.dec4 = nn.ConvTranspose2d(512, 256, 2, 2), ConvBlock(512, 256)
        self.up3, self.dec3 = nn.ConvTranspose2d(256, 128, 2, 2), ConvBlock(256, 128)
        self.up2, self.dec2 = nn.ConvTranspose2d(128, 64, 2, 2), ConvBlock(128, 64)
        self.up1, self.dec1 = nn.ConvTranspose2d(64, 32, 2, 2), ConvBlock(96, 32)
        self.up0, self.head = nn.ConvTranspose2d(32, 16, 2, 2), nn.Conv2d(16, 1, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, image):
        image = (image - self.mean) / self.std
        x0 = self.stem(image)
        x1 = self.layer1(self.pool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        x = self.dec4(torch.cat([self.up4(x4), x3], 1))
        x = self.dec3(torch.cat([self.up3(x), x2], 1))
        x = self.dec2(torch.cat([self.up2(x), x1], 1))
        x = self.dec1(torch.cat([self.up1(x), x0], 1))
        return self.head(self.up0(x))

In [ ]:
best_model = YOLO("best_model.pt")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


In [ ]:
# ============================================================
# Inference: YOLO finds instacnes -> refiner adds details
# ============================================================

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
refiner = CropUNet(pretrained=False)
refiner.load_state_dict(torch.load("best_refiner.pt", map_location=device))
refiner.to(device).eval()
REFINER_THRESHOLD = 0.65


@torch.no_grad()
def refine_masks(image_gray, yolo_masks, refiner, device, threshold=REFINER_THRESHOLD):
    refined = []
    for mask in yolo_masks:
        if mask.sum() == 0:
            refined.append(mask)
            continue
        x0, y0, x1, y1 = square_bounds(mask)
        crop = cv2.resize(image_gray[y0:y1, x0:x1], (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_LINEAR)
        tensor = torch.from_numpy(crop).float().unsqueeze(0).unsqueeze(0).repeat(1, 3, 1, 1).to(device) / 255
        prob = refiner(tensor).sigmoid()[0, 0].cpu().numpy()
        crop_mask = cv2.resize(prob, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR) >= threshold

        if crop_mask.any():
            full = np.zeros_like(mask)
            full[y0:y1, x0:x1] = crop_mask
            refined.append(full)
        else:
            refined.append(mask)

    return refined

In [ ]:
import pandas as pd
from pycocotools import mask as mask_utils

refiner = CropUNet(pretrained=False)
refiner.load_state_dict(torch.load("best_refiner.pt", map_location=device))
refiner.to(device).eval()

test_files = sorted([
    f for f in os.listdir(CFG.TEST_DIR)
    if f.endswith(".jpeg")
])

rows = []

for filename in tqdm(test_files, desc="YOLO + refiner inference"):

    image_path = os.path.join(CFG.TEST_DIR, filename)
    image_gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    r = best_model.predict(
        image_path,
        imgsz=CFG.IMGSZ,
        conf=CFG.CONF_THRESHOLD,
        agnostic_nms=True,
        retina_masks=True,
        verbose=False
    )[0]

    if r.masks is None:
        continue

    yolo_masks = [m.cpu().numpy().astype(np.uint8) for m in r.masks.data]

    refined = refine_masks(image_gray, yolo_masks, refiner, device)

    image_id = filename.replace(".jpeg", "")
    counter = 1
    occupied = np.zeros_like(image_gray, dtype=bool)

    for mask in refined:
        mask = np.asarray(mask, dtype=bool).copy()
        mask[occupied] = False

        if mask.sum() < CFG.MIN_COMPONENT_SIZE:
            continue

        occupied |= mask
        rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))

        rows.append({
            "filament_id": f"{image_id}_{counter}",
            "segmentation_rle": rle["counts"].decode("utf-8")
        })

        counter += 1

submission = pd.DataFrame(rows)
submission.to_csv("submission.csv", index=False)

print(submission.head())
print(f"Total predicted filaments: {len(submission)}")

In [ ]:
assert list(submission.columns) == ["filament_id", "segmentation_rle"]
assert not submission.empty
assert submission["filament_id"].is_unique
assert submission["segmentation_rle"].map(lambda value: isinstance(value, str) and bool(value)).all()
known_images = {Path(name).stem for name in test_files}
submitted_images = {value.rsplit("_", 1)[0] for value in submission["filament_id"]}
assert submitted_images <= known_images
overlap_violations = 0
for _, group in submission.groupby(submission["filament_id"].str.rsplit("_", n=1).str[0], sort=False):
    occupied_rle = None
    for counts in group["segmentation_rle"]:
        current_rle = {"size": [2048, 2048], "counts": counts.encode("ascii")}
        assert int(mask_utils.area(current_rle)) > 0
        if occupied_rle is not None:
            intersection = mask_utils.merge([occupied_rle, current_rle], intersect=True)
            overlap_violations += int(mask_utils.area(intersection) > 0)
            occupied_rle = mask_utils.merge([occupied_rle, current_rle], intersect=False)
        else:
            occupied_rle = current_rle
assert overlap_violations == 0
report = {
    "source": PUBLIC_HANDLE,
    "source_license": "Apache-2.0",
    "asset_sha256": EXPECTED,
    "test_images": len(known_images),
    "images_with_predictions": len(submitted_images),
    "prediction_rows": len(submission),
    "overlap_violations": overlap_violations,
    "missing_prediction_images": sorted(known_images - submitted_images),
}
Path("public-yolo-unet-inference-report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
